In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import datetime
import requests
import pandas as pd

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MO MAM' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running MO MAM Web Scraping Tool v.1.0


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

processdate = now.strftime('%Y-%m-%d')

# Lists from Jira DECD-6135. The site is a Vue SPA - page URLs are an empty shell.
# All data comes from the unauthenticated JSON API:
#   GET https://www.amcm.gov.mo/api/v1.0/cms/companies?slug=<SLUG>   (header Api-Language: en)
# The ?type= param in the site URLs equals the API slug 1:1.
regdict = {
    1: {"ListName": "Banks incorporated in Macau",
        "URL": "https://www.amcm.gov.mo/en/bank/bank-institutions-list?type=company-parent-bank-leaf-1"},
    2: {"ListName": "Branches of banks incorporated overseas",
        "URL": "https://www.amcm.gov.mo/en/bank/bank-institutions-list?type=company-parent-bank-leaf-2"},
    3: {"ListName": "Macao Postal Savings (Caixa Económica Postal)",
        "URL": "https://www.amcm.gov.mo/en/bank/bank-institutions-list?type=company-parent-bank-leaf-3"},
    4: {"ListName": "Finance companies",
        "URL": "https://www.amcm.gov.mo/en/other-institution/other-institution-institutions-list?type=company-parent-other-institution-1"},
    5: {"ListName": "Other Financial Institutions",
        "URL": "https://www.amcm.gov.mo/en/other-institution/other-institution-institutions-list?type=company-parent-other-institution-7"},
    6: {"ListName": "Insurance Institutions",
        "URL": "https://www.amcm.gov.mo/en/insurance-sector/insurance-sector-institutions-list/company-parent-insurance-1?type=company-parent-insurance-1"},
}

# per-list API slugs; lists 5/6 span several subsections (subsection title -> CoType)
listslugs = {
    1: [('company-parent-bank-leaf-1', '')],
    2: [('company-parent-bank-leaf-2', '')],
    3: [('company-parent-bank-leaf-3', '')],
    4: [('company-parent-other-institution-1', '')],
    5: [('company-parent-other-institution-3', 'Remittance companies'),
        ('company-parent-other-institution-4', 'Money changers'),
        ('company-parent-other-institution-9', 'Financial leasing companies'),
        ('company-parent-other-institution-7', 'Financial asset trading companies'),
        ('company-parent-other-institution-8', 'Payment services'),
        ('company-parent-other-institution-6', 'Securities intermediaries'),
        ('company-parent-other-institution-2', 'Others')],
    6: [('company-parent-insurance-1', 'Life Insurers'),
        ('company-parent-insurance-2', 'General Insurers'),
        # ('company-parent-insurance-3', 'Insurers & Insurance Intermediaries Associations'),
        ('company-parent-insurance-5', 'Representative Offices')],
}

ListLabeldict = {1: 1, 2: 1, 3: 1, 4: 4, 5: 4, 6: 2}

# 'Head Office: <place>' in the branch field -> home country of the mother institution
mother_iso = {'Bermuda': 'BM', 'Canada': 'CA', 'Hong Kong': 'HK', 'Macau': 'MO',
              'P.R.C.': 'CN', 'United States of America': 'US'}

API = 'https://www.amcm.gov.mo/api/v1.0/cms/companies'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
           'Api-Language': 'en'}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def fetch_companies(slug):
    r = requests.get(API, params={'slug': slug}, headers=HEADERS, timeout=60, verify=False)
    r.raise_for_status()
    data = r.json().get('data') or {}   # some slugs return "data": null
    updated = (data.get('updatedAt') or '')[:10]
    return data.get('companies') or [], updated

def add_row(rowdata):
    for key in sqldict:
        sqldict[key].append(rowdata.get(key, ''))

def common_fields(listnr):
    return {'ListLabel': ListLabeldict[listnr],
            'RegCtry': 'MO',
            'RegCode': 'MAM',
            'ListCode': str(listnr),
            'ListName': regdict[listnr]['ListName'],
            'ListLanguage': 'EN',
            'ListProcessDate': processdate,
            'RegulationType': 'Regulated',
            'Cntry': 'MO'}

def clean_email(email):
    email = (email or '').strip()
    if email.startswith('mailto:'):
        email = email[len('mailto:'):]
    return email.strip()

In [5]:
#------------------------------------------------ Begin_Main : all 6 lists via the companies API ----------------------------------------
for listnr, slugpairs in listslugs.items():
    print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")
    count = 0
    for slug, subsection in slugpairs:
        companies, updated = fetch_companies(slug)
        if not companies:
            print(f"[WARN] : slug '{slug}' returned no companies")
        for c in companies:
            branch = (c.get('branch') or '').strip()
            mother = ''
            if branch.startswith('Head Office:'):
                place = branch.split(':', 1)[1].strip()
                mother = mother_iso.get(place, place)   # keep verbatim if not in the map
            row = common_fields(listnr)
            row.update({'Name': (c.get('company') or '').strip(),
                        'CoType': subsection,
                        'Address_1': (c.get('address') or '').strip(),
                        'Phone': (c.get('tel') or '').strip(),
                        'Fax': (c.get('fax') or '').strip(),
                        'Email': clean_email(c.get('email')),
                        'Website': (c.get('website') or '').strip(),
                        'ListValidityDate': updated,
                        'Cntry - Mother company': mother})
            add_row(row)
            count += 1
    print(f"[INFO] : List {listnr} -> {count} entities")

[INFO] : Working _(Banks incorporated in Macau)_ 


[INFO] : List 1 -> 12 entities
[INFO] : Working _(Branches of banks incorporated overseas)_ 


[INFO] : List 2 -> 19 entities
[INFO] : Working _(Macao Postal Savings (Caixa Económica Postal))_ 


[INFO] : List 3 -> 1 entities
[INFO] : Working _(Finance companies)_ 


[INFO] : List 4 -> 1 entities
[INFO] : Working _(Other Financial Institutions)_ 


[INFO] : List 5 -> 31 entities
[INFO] : Working _(Insurance Institutions)_ 


[INFO] : List 6 -> 34 entities


In [6]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 98 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/MO MAM/MO MAM SQL Ready 2026-07-13 10.44.11.xlsx
